<a href="https://colab.research.google.com/github/DQ0115/1142programing_language/blob/main/HW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-generativeai

In [2]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
from google.colab import userdata
from google import genai

# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
client = genai.Client(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash'

In [4]:
response = client.models.generate_content(
    model = MODEL_ID, contents="Explain how AI works in a few words"
)
print(response.text)

AI learns patterns from data to make decisions or predictions.


In [11]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1_jJnmcuHIvAkS0eRO4Z53MQlRdflaF6AXYeKAO3BluY/edit?gid=0#gid=0"
WORKSHEET_NAME = "工作表1"

REQUIRED_COLUMNS = ["日期", "科目", "作業成績"]

_auth_done = False
_gc = None
_ws = None

In [6]:
# --- 主要功能區塊 ---
def get_user_grades():
    """
    透過終端機輸入學生成績，直到使用者輸入 'q' 結束。
    """
    print("--- 準備輸入成績。輸入 'q' 來停止。---")
    grades = []
    while True:
        subject = input("請輸入科目（或輸入 'q' 停止）：")
        if subject.lower() == 'q':
            break

        grade = input(f"請輸入 {subject} 的成績：")
        try:
            grade = int(grade)
        except ValueError:
            print("成績必須是數字。請重新輸入。")
            continue

        today = datetime.now().strftime('%Y-%m-%d')
        grades.append([today, subject, grade])
        print(f"已記錄：日期: {today}, 科目: {subject}, 成績: {grade}\n")

    return grades

In [7]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = client.models.generate_content(model = MODEL_ID, contents = prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

In [8]:
new_grades = get_user_grades()

--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：國文
請輸入 國文 的成績：100
已記錄：日期: 2026-04-22, 科目: 國文, 成績: 100

請輸入科目（或輸入 'q' 停止）：數學
請輸入 數學 的成績：85
已記錄：日期: 2026-04-22, 科目: 數學, 成績: 85

請輸入科目（或輸入 'q' 停止）：自然
請輸入 自然 的成績：75
已記錄：日期: 2026-04-22, 科目: 自然, 成績: 75

請輸入科目（或輸入 'q' 停止）：q


In [9]:
new_grades

[['2026-04-22', '國文', 100], ['2026-04-22', '數學', 85], ['2026-04-22', '自然', 75]]

In [10]:
get_ai_summary(new_grades)


--- 正在呼叫 AI 模型生成摘要... ---


'好的，這是一份根據您提供的學生單次成績列表所產生的簡單摘要與常見迷思整理，其中不包含任何評分或主觀判斷。\n\n---\n\n### **學生單次成績摘要 (2026-04-22)**\n\n這份成績顯示了學生在2026年4月22日，於三個科目中的一次測驗表現：\n\n*   **國文：** 100分 (滿分)\n*   **數學：** 85分\n*   **自然：** 75分\n\n**總結：**\n學生在這次測驗中，國文科目表現出色，獲得滿分。數學科目獲得85分，自然科目獲得75分。整體而言，三科成績分佈於75分至100分之間。\n\n---\n\n### **關於成績的常見迷思整理 (不評分，只做總結)**\n\n以下整理了一些在解讀學生單次成績時，可能存在的常見迷思，旨在提供更全面的視角，而非對學生進行評價：\n\n1.  **迷思一：單次成績完全代表學生的能力或潛力。**\n    *   **實際情況：** 單次成績是學生在特定時間、特定情境下（如考試當天的身體狀況、心情、題目難易度、準備程度、考前壓力等）的學習成果呈現。它是一個「快照」，而非學生長期學習軌跡或總體能力的完整寫照。不同科目間的成績差異，也可能受學習興趣、教學方式適應度、教材內容深淺等因素影響。\n\n2.  **迷思二：不同科目的相同分數具有相同的意義。**\n    *   **實際情況：** 儘管都是分數，但不同科目的評量標準、知識結構、思考模式和學習方法可能大相徑庭。例如，國文滿分可能反映了對語文的良好理解與表達，而自然75分可能代表對科學概念的初步掌握。直接比較分數高低，而忽略科目本質差異，可能會產生偏頗的解讀。\n\n3.  **迷思三：高分代表所有內容都已精通，低分則一無是處。**\n    *   **實際情況：** 高分固然值得肯定，但學生可能仍有某些細節或應用層面需要加強。同樣地，即使分數較低，學生也可能在某些部分有著紮實的理解，只是在特定測驗中未能完全展現或有特定弱項。分數背後所反映的「哪些知識點掌握了？哪些是模糊的？錯誤的類型是什麼？」這些學習過程中的細節，往往比單一數字更有價值。\n\n4.  **迷思四：成績是判斷學生學習態度好壞的唯一標準。**\n    *   **實際情況：** 學習態度、努力程度和學習成效之間確實存在關聯，但並非絕對的因果關係。一位學生

In [12]:
def main():
    """
    主程式流程：輸入成績 -> 獲取 AI 摘要 -> 寫入 Google Sheet。
    """
    try:
        # 1. Google Sheet 身份驗證
        auth.authenticate_user()

        creds, _ = default()
        gc = gspread.authorize(creds)

        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)





        print("--- Google Sheet 連線成功。---")

        # 2. 獲取使用者輸入的成績
        new_grades = get_user_grades()

        if not new_grades:
            print("沒有輸入任何成績，程式結束。")
            return

        # 3. 將新成績寫入 Google Sheet
        ws.append_rows(new_grades)
        print("\n--- 成績已成功寫入 Google Sheet。---")

        # 4. 獲取 AI 摘要並寫入 Google Sheet
        summary = get_ai_summary(new_grades)

        # 尋找第一行空白列
        next_row = len(ws.col_values(1)) + 1

        # 使用 update_cell() 方法逐一更新儲存格
        ws.update_cell(next_row, 1, datetime.now().strftime('%Y-%m-%d'))
        ws.update_cell(next_row, 2, 'AI 摘要')

        # 為了避免單元格內容過長，將摘要內容分成多行來寫入
        summary_lines = summary.split('\n')
        for i, line in enumerate(summary_lines):
            ws.update_cell(next_row + i, 3, line)

        print("\n--- AI 摘要已成功寫入 Google Sheet。---")
        print("以下是 AI 生成的摘要內容：")
        print("-" * 50)
        print(summary)
        print("-" * 50)

    except gspread.exceptions.APIError as e:
        print(f"Google Sheets API 錯誤：{e.response.text}")
        print("請確認：")
        print("1. 您的服務帳戶金鑰檔案正確且未過期。")
        print("2. 您已將服務帳戶的 Email 地址（在 JSON 檔案中）分享給 Google Sheet，並給予編輯權限。")
    except Exception as e:
        print(f"發生未預期的錯誤：{e}")

if __name__ == "__main__":
    main()

--- Google Sheet 連線成功。---
--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：國文
請輸入 國文 的成績：100
已記錄：日期: 2026-04-22, 科目: 國文, 成績: 100

請輸入科目（或輸入 'q' 停止）：數學
請輸入 數學 的成績：85
已記錄：日期: 2026-04-22, 科目: 數學, 成績: 85

請輸入科目（或輸入 'q' 停止）：自然
請輸入 自然 的成績：75
已記錄：日期: 2026-04-22, 科目: 自然, 成績: 75

請輸入科目（或輸入 'q' 停止）：q

--- 成績已成功寫入 Google Sheet。---

--- 正在呼叫 AI 模型生成摘要... ---

--- AI 摘要已成功寫入 Google Sheet。---
以下是 AI 生成的摘要內容：
--------------------------------------------------
好的，這是一份根據您提供的成績資料所做的摘要與常見迷思整理，全程不帶任何評分判斷。

---

### 成績摘要

以下是針對2026年4月22日所列出的科目成績的客觀總結：

*   **評量日期：** 所有科目成績均顯示於2026年4月22日。
*   **科目數量：** 共包含三個科目。
*   **科目與成績分佈：**
    *   國文：100分
    *   數學：85分
    *   自然：75分
*   **成績範圍：** 三個科目的成績分佈從75分到100分。
*   **最高與最低：** 國文科目獲得最高分100分，自然科目獲得最低分75分。
*   **平均分數：** 三個科目的平均分數為 (100 + 85 + 75) / 3 = 約 86.67分。

---

### 關於成績的常見迷思與理解

成績是學習過程中的一個重要反饋工具，但圍繞著成績的意義和作用，常有一些常見的迷思。以下將對這些迷思進行整理與澄清：

1.  **迷思一：成績是衡量學生能力的唯一或最佳標準。**
    *   **理解：** 成績反映了學生在特定時間、特定科目中的知識掌握程度、理解能力以及應試技巧。它是評估學習進度的一種重要方式，但學生的整體能力、潛力、創造力、解決問題能力、批判性思維